In [8]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from kaggle.api.kaggle_api_extended import KaggleApi
from datetime import datetime
from pathlib import Path
import tempfile, zipfile
from datetime import datetime
from streamlit_autorefresh import st_autorefresh
import pandas as pd

In [23]:
current.reset_index(drop=True,inplace=True)

In [15]:
api = get_api()
current = fetch_leaderboard(api)
history= load_history()
pd.concat([history,current],axis=0)

C:\Users\Kacper\AppData\Local\Temp\ipykernel_34368\4238277851.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([history,current],axis=0)


,TeamName,Rank,Score,FetchDate
24,Ryszard Czarnecki,25,0.08615,2026-03-21 02:43:43.057284
500,Kacper Rzeźniczak,501,0.09953,2026-03-21 02:43:43.057284
1380,Norbert Gościcki,1381,0.10947,2026-03-21 02:43:43.057284
1394,Stefan Gajda,1395,0.10975,2026-03-21 02:43:43.057284


In [13]:
# --------------- CONFIG ---------------

# odświeżaj co 10 minut (600000 ms)
st_autorefresh(interval=10 * 60 * 1000, key="refresh")

COMPETITION = "march-machine-learning-mania-2026"

     # zmień jeśli inna nazwa
USERNAMES = [
    "Kacper Rzeźniczak",      # <- wpisz swoje username'y
    "Ryszard Czarnecki",
    "Stefan Gajda",
    "Norbert Gościcki",
    #"kolega4",
]
HISTORY_FILE = Path("score_history.csv")
# --------------------------------------


@st.cache_resource
def get_api():
    api = KaggleApi()
    api.authenticate()
    return api


def fetch_leaderboard(api) -> pd.DataFrame:
    """Pobiera leaderboard i filtruje po naszych username'ach."""
    global COMPETITION
    with tempfile.TemporaryDirectory() as tmp:
        api.competition_leaderboard_download(COMPETITION,tmp)
        with zipfile.ZipFile(f"{tmp}/{COMPETITION}.zip") as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f)
    date_now=datetime.now()
    df = df.loc[df["TeamName"].isin(USERNAMES),['TeamName','Rank','Score']]
    df['FetchDate'] = date_now
    return df


def load_history() -> pd.DataFrame:
    """Ładuje historię score'ów z CSV."""
    if HISTORY_FILE.exists():
        return pd.read_csv(HISTORY_FILE, parse_dates=["timestamp"])
    return pd.DataFrame(columns=['TeamName','Rank','Score','FetchDate'])


def save_snapshot(current: pd.DataFrame, history: pd.DataFrame) -> pd.DataFrame:
    """Dodaje aktualny snapshot do historii."""
    if current.equals(history.loc[-4:,:]):
        return history
    updated = pd.concat([history, current], ignore_index=True)
    updated.to_csv(HISTORY_FILE, index=False)
    return updated


# --------------- UI ---------------
st.set_page_config(page_title="March Madness Tracker", layout="wide")
st.title("🏀 March Madness — Score Tracker")

api = get_api()

col1, col2 = st.columns([1, 3])

with col1:
    st.subheader("⚙️ Panel")
    if st.button("🔄 Pobierz nowy snapshot", type="primary", use_container_width=True):
        with st.spinner("Pobieram leaderboard z Kaggle..."):
            current = fetch_leaderboard(api)
            if current.empty:
                st.error("Nie znaleziono żadnego z username'ów na leaderboardzie. Sprawdź USERNAMES i COMPETITION w kodzie.")
            else:
                history = load_history()
                history = save_snapshot(current, history)
                st.success(f"Zapisano snapshot ({len(current)} graczy)")
                st.rerun()

    st.markdown("---")
    st.caption("Snapshoty zapisują się do `score_history.csv`. Możesz odpalać przycisk po każdym meczu.")

# Load history for display
history = load_history()

with col2:
    if history.empty:
        st.info("Brak danych — kliknij **Pobierz nowy snapshot** żeby zacząć śledzić.")
    else:
        # --- Aktualny ranking ---
        st.subheader("📊 Aktualny ranking")
        latest_ts = history["timestamp"].max()
        latest = (
            history[history["timestamp"] == latest_ts]
            .sort_values("rank")
            .reset_index(drop=True)
        )
        latest.index += 1
        st.dataframe(
            latest[["username", "score", "rank"]].rename(
                columns={"username": "Gracz", "score": "Log Loss", "rank": "Pozycja"}
            ),
            use_container_width=True,
        )

        # --- Wykres score w czasie ---
        st.subheader("📈 Score w czasie")
        fig_score = px.line(
            history,
            x="timestamp",
            y="score",
            color="username",
            markers=True,
            labels={"timestamp": "Czas", "score": "Log Loss", "username": "Gracz"},
        )
        fig_score.update_layout(
            yaxis_title="Log Loss (niżej = lepiej)",
            hovermode="x unified",
        )
        st.plotly_chart(fig_score, use_container_width=True)

        # --- Wykres rank w czasie ---
        st.subheader("🏆 Pozycja w czasie")
        fig_rank = px.line(
            history,
            x="timestamp",
            y="rank",
            color="username",
            markers=True,
            labels={"timestamp": "Czas", "rank": "Pozycja", "username": "Gracz"},
        )
        fig_rank.update_layout(
            yaxis_title="Pozycja (niżej = lepiej)",
            yaxis_autorange="reversed",
            hovermode="x unified",
        )
        st.plotly_chart(fig_rank, use_container_width=True)

        # --- Tabela zmian ---
        if len(history["timestamp"].unique()) >= 2:
            st.subheader("🔀 Zmiany od ostatniego snapshotu")
            timestamps = sorted(history["timestamp"].unique())
            prev_ts = timestamps[-2]
            prev = history[history["timestamp"] == prev_ts].set_index("username")
            curr = latest.set_index("Gracz")  # renamed column

            deltas = []
            for user in USERNAMES:
                if user in prev.index and user in curr.index:
                    score_delta = curr.loc[user, "Log Loss"] - prev.loc[user, "score"]
                    rank_delta = curr.loc[user, "Pozycja"] - prev.loc[user, "rank"]
                    deltas.append({
                        "Gracz": user,
                        "Δ Score": f"{score_delta:+.4f}",
                        "Δ Pozycja": f"{int(rank_delta):+d}" if rank_delta != 0 else "—",
                    })
            if deltas:
                st.dataframe(pd.DataFrame(deltas), use_container_width=True)


2026-03-21 02:43:38.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 02:43:38.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
lb

[{"teamId": 15299718, "teamName": "Hu\u1ef3nh Qu\u1ed1c Vi\u1ec7t", "submissionDate": "2026-02-25T09:30:02.260Z", "score": "0.06577"},
 {"teamId": 15352937, "teamName": "Guest_User66", "submissionDate": "2026-03-07T13:14:50.970Z", "score": "0.07731"},
 {"teamId": 15337621, "teamName": "lookahead bias and pray", "submissionDate": "2026-03-16T19:49:52.096Z", "score": "0.08286"},
 {"teamId": 15334924, "teamName": "Shril Patel", "submissionDate": "2026-03-03T20:19:23.443Z", "score": "0.08303"},
 {"teamId": 15295819, "teamName": "KiroSamurai", "submissionDate": "2026-03-17T06:42:51.833Z", "score": "0.08375"},
 {"teamId": 15357083, "teamName": "Avarch", "submissionDate": "2026-03-19T10:00:12.880Z", "score": "0.08773"},
 {"teamId": 15415346, "teamName": "Ethan Duncan", "submissionDate": "2026-03-18T18:30:59.133Z", "score": "0.08874"},
 {"teamId": 15393248, "teamName": "Marcus Harvey", "submissionDate": "2026-03-18T11:55:49.356Z", "score": "0.08891"},
 {"teamId": 15279660, "teamName": "Nguy\u1

In [ ]:
def get_leaderboard(api):
    global COMPETITION
    with tempfile.TemporaryDirectory() as tmp:
        api.competition_leaderboard_download(COMPETITION,tmp)
        with zipfile.ZipFile(f"{tmp}/{COMPETITION}.zip") as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f)
    return df

In [ ]:
df

,Rank,TeamId,TeamName,LastSubmissionDate,Score,SubmissionCount,TeamMemberUserNames
0,1,15299718,Huỳnh Quốc Việt,2026-02-25 09:30:02,0.06577,1,hunhqucvit
1,2,15352937,Guest_User66,2026-03-07 13:14:50,0.07731,1,guestuser66
2,3,15337621,lookahead bias and pray,2026-03-16 19:49:52,0.08286,1,prospal
3,4,15334924,Shril Patel,2026-03-03 20:19:23,0.08303,1,zerol0l
4,5,15295819,KiroSamurai,2026-03-17 06:42:51,0.08375,1,kirosamurai
...,...,...,...,...,...,...,...
3480,3481,15392135,bobbyd73,2026-03-15 16:48:34,0.57375,1,bobbyd73
3481,3482,15393834,Ramana Prabhu Sana,2026-03-16 21:25:18,0.59256,1,ramanaprabhusana
3482,3483,15315782,SuperDubby,2026-03-02 07:53:04,0.59363,1,superdubby
3483,3484,15280222,SHWETA UMBRAJKAR,2026-02-23 05:58:04,0.63468,1,shwetaumbrajkar


In [ ]:
df.loc[:,"TeamName"].isin(USERNAMES)

0       False
1       False
2       False
3       False
4       False
        ...  
3480    False
3481    False
3482    False
3483    False
3484    False
Name: TeamName, Length: 3485, dtype: bool

time.struct_time(tm_year=2026, tm_mon=3, tm_mday=21, tm_hour=2, tm_min=4, tm_sec=48, tm_wday=5, tm_yday=80, tm_isdst=0)


In [ ]:
COMPETITION = "march-machine-learning-mania-2026"
api = KaggleApi()
api.authenticate()
lb = api.competition_leaderboard_view(COMPETITION)
type(lb)
# for en in lb:
#     print(type(en))
#     print(en)
#     print(f"teamID {en.team_id}")
[m for m in dir(KaggleApi) if not m.startswith('_')]
# lb = api.competition_leaderboard_view(competition=)
def get_leaderboard(api):
    global COMPETITION
    with tempfile.TemporaryDirectory() as tmp:
        api.competition_leaderboard_download(COMPETITION,tmp)
        with zipfile.ZipFile(f"{tmp}/{COMPETITION}.zip") as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f)
    date_now=datetime.now()
    df = df.loc[df["TeamName"].isin(USERNAMES),['TeamName','Rank','Score']]
    df['get_date'] = date_now
    return df


Next Page Token = CfDJ8L9-_gcGHthJtufiBgcc1obPuQEh49R5jQ5xm0jfSl72xufgmrqTN0u-P45LwPj7-yjfbxo8HJBe4dannLlPDAQ


In [ ]:
df

,TeamName,Rank,Score,get_date
23,Ryszard Czarnecki,24,0.09632,2026-03-21 02:07:17.904062
519,Kacper Rzeźniczak,520,0.11186,2026-03-21 02:07:17.904062
1410,Stefan Gajda,1411,0.12256,2026-03-21 02:07:17.904062
1421,Norbert Gościcki,1422,0.12277,2026-03-21 02:07:17.904062


In [ ]:
df.iloc[-3:,:]

,TeamName,Rank,Score,get_date
519,Kacper Rzeźniczak,520,0.11186,2026-03-21 02:07:17.904062
1410,Stefan Gajda,1411,0.12256,2026-03-21 02:07:17.904062
1421,Norbert Gościcki,1422,0.12277,2026-03-21 02:07:17.904062
